In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, from_json, current_timestamp, to_timestamp
from config import PATHS, CHECKPOINTS
from metrics_utils import log_pipeline_metric
from pyspark.sql.types import *

In [0]:
EH_NAMESPACE = "ehns-retail"
EH_HUB_NAME = "transactions"

BOOTSTRAP_SERVERS = f"{EH_NAMESPACE}.servicebus.windows.net:9093"

CONNECTION_STRING = "Endpoint=sb://ehns-retail.servicebus.windows.net/;SharedAccessKeyName=spark-consumer;SharedAccessKey=XXXX"

In [0]:
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "subscribe": EH_HUB_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{CONNECTION_STRING}";',
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

In [0]:
schema = StructType([
    StructField("InvoiceNo", StringType()),
    StructField("StockCode", StringType()),
    StructField("Description", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("InvoiceDate", StringType()),
    StructField("UnitPrice", DoubleType()),
    StructField("CustomerID", DoubleType()),
    StructField("Country", StringType())
])

In [0]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

In [0]:
df_stream = (
    df_kafka
    .selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), schema).alias("data"))
    .select("data.*")
    .withColumn("invoice_ts", to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    .withColumn("_ingest_ts", current_timestamp())
)

In [0]:
df_stream = df_stream.drop("_rescued_data")

In [0]:
from pyspark.sql import functions as F

def log_pipeline_metric(spark, pipeline_name, df):

    metrics_df = spark.createDataFrame(
        [(pipeline_name, df.count())],
        ["pipeline", "row_count"]
    ).withColumn("processed_at", F.current_timestamp())

    metrics_df.write.mode("append").saveAsTable(
        "dbw_retails.monitoring.pipeline_metrics"
    )

In [0]:
def process_batch(batch_df, batch_id):

    from config import TABLES
    from metrics_utils import log_pipeline_metric

    batch_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLES["bronze_eventhub"])

    log_pipeline_metric(spark, "bronze_eventhub", batch_df)

In [0]:
query = (
    df_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", PATHS["bronze_eventhub"] + "_checkpoint")
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()